# Projekt 01 (basic) — Mehrsprachige Subword-Tokenisierung mit SentencePiece

**Modul 10 — Multilingual NLP** · Format: **Jupyter Notebook** (`tokenisierung.ipynb`)

Bevor ein mehrsprachiges Modell auch nur ein Gewicht lernt, muss Text in Tokens zerlegt
werden — und *wie* man das tut, entscheidet über OOV-Rate, Sequenzlänge und darüber, ob
Sprachen sich Repräsentationen **teilen** können. Dieses Projekt macht die Tokenisierung
greifbar: Du trainierst ein **gemeinsames Subword-Vokabular** auf Deutsch **und** Englisch
und misst empirisch, was der Skript-Abschnitt 2 behauptet.

Kernfragen, die du beantwortest:

- Wie sieht eine **BPE**-Zerlegung aus, und wie geht sie mit unbekannten Wörtern um?
- Was ist **fertility** (Ø Subword-Tokens pro Wort), und wie unterscheidet sie sich
  zwischen Sprachen?
- Wie stark **teilen** sich Deutsch und Englisch Tokens in einem gemeinsamen Vokabular?
- Was passiert, wenn das Vokabular **englisch-dominiert** ist (der *fairness*-Aspekt
  großer LLMs)?

> **Viel Anleitung:** Download, Parsing und die SentencePiece-Aufrufe sind vorgegeben.
> Deine Aufgaben treffen die Analyse-Kerne: fertility berechnen, Vokabular-Teilung messen
> und den Vokabular-Bias experimentell zeigen.


## Setup

Benötigt `sentencepiece` (in der Repo-`requirements.txt`). Die erste Zelle lädt den
**Tatoeba** Deutsch–Englisch-Satzpaar-Datensatz (~12 MB) nach `daten/` und cached ihn.

```bash
source ../../../../.venv/bin/activate
jupyter lab      # oder tokenisierung.ipynb in VS Code öffnen, Kernel = Repo-.venv
```

Alles läuft in **unter einer Minute** (SentencePiece ist in C++ und sehr schnell).


## Teil A — Daten laden & Korpus vorbereiten *(vorgegeben)*

**Tatoeba** ist eine offene Sammlung von Übersetzungs-Satzpaaren. Wir nutzen die
Deutsch–Englisch-Paare (Format pro Zeile: `englisch \t deutsch \t Attribution`). Zum
schnellen Training schreiben wir die ersten Zehntausend Paare in eine Korpus-Datei.


In [1]:
# ---- Tatoeba DE-EN laden (echte Uebersetzungspaare) -----------------------
import os, io, zipfile, urllib.request

DATA_DIR = "daten"
os.makedirs(DATA_DIR, exist_ok=True)
RAW = os.path.join(DATA_DIR, "deu.txt")
URL = "https://www.manythings.org/anki/deu-eng.zip"

if not os.path.exists(RAW):
    print("Lade Tatoeba DE-EN ...")
    # Die Seite verlangt Browser-artige Header (sonst HTTP 406).
    hdr = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
           "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
           "Accept-Language": "en-US,en;q=0.9",
           "Referer": "https://www.manythings.org/anki/"}
    req = urllib.request.Request(URL, headers=hdr)
    raw = urllib.request.urlopen(req, timeout=60).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with z.open("deu.txt") as src, open(RAW, "wb") as dst:
            dst.write(src.read())
    print("Fertig.")
else:
    print("Datensatz bereits vorhanden.")

lines = open(RAW, encoding="utf-8").read().strip().split("\n")
pairs = [ln.split("\t")[:2] for ln in lines]        # [englisch, deutsch]
print(f"{len(pairs):,} Satzpaare.")
for en, de in pairs[500:503]:
    print(f"  EN: {en}\n  DE: {de}\n")


Datensatz bereits vorhanden.
331,266 Satzpaare.
  EN: I'm bald.
  DE: Ich habe eine Glatze.

  EN: I'm busy.
  DE: Ich bin beschäftigt.

  EN: I'm busy.
  DE: Ich habe zu tun.



In [2]:
# ---- Korpus-Datei fuer SentencePiece schreiben (EN + DE gemischt) ----------
N = 40000                                     # Teilmenge fuer schnelles Training
subset = pairs[:N]
en_sents = [en for en, de in subset]
de_sents = [de for en, de in subset]

CORPUS = os.path.join(DATA_DIR, "corpus_deen.txt")
with open(CORPUS, "w", encoding="utf-8") as f:
    for en, de in subset:
        f.write(en + "\n")
        f.write(de + "\n")
print(f"Korpus geschrieben: {2*N:,} Zeilen ({N:,} EN + {N:,} DE)")


Korpus geschrieben: 80,000 Zeilen (40,000 EN + 40,000 DE)


### Aufgabe 1 — Ein gemeinsames BPE-Vokabular trainieren

Der Trainingsaufruf ist vorgegeben. Trainiere ein **gemeinsames** BPE-Modell (Vokabular
8000) auf dem gemischten DE+EN-Korpus und schau dir die Zerlegungen an.

**Deine Aufgabe (`# TODO`):** Nutze den geladenen Prozessor `sp`, um die beiden
Beispielsätze zu **encoden** (`sp.encode(text, out_type=str)`), und gib die Token-Listen
aus. Achte auf `▁` — das ist SentencePieces Markierung für ein Leerzeichen (Wortanfang).


In [3]:
# ---- BPE-Modell trainieren (vorgegeben) + encoden (DU) --------------------
import sentencepiece as spm

MODEL_PREFIX = os.path.join(DATA_DIR, "shared_bpe")
if not os.path.exists(MODEL_PREFIX + ".model"):
    spm.SentencePieceTrainer.train(
        input=CORPUS, model_prefix=MODEL_PREFIX,
        vocab_size=8000, model_type="bpe",
        character_coverage=1.0, input_sentence_size=200000,
        shuffle_input_sentence=True)
sp = spm.SentencePieceProcessor(model_file=MODEL_PREFIX + ".model")
print("Vokabulargroesse:", sp.get_piece_size())

ex_en = "I don't understand this complicated sentence."
ex_de = "Ich verstehe diesen komplizierten Satz nicht."
# TODO: beide Saetze mit sp.encode(..., out_type=str) zerlegen und ausgeben
print("EN:", sp.encode(ex_en, out_type=str))
print("DE:", sp.encode(ex_de, out_type=str))


Vokabulargroesse: 8000
EN: ['▁I', '▁don', "'", 't', '▁understand', '▁this', '▁com', 'pl', 'icated', '▁sen', 'ten', 'ce', '.']
DE: ['▁Ich', '▁verstehe', '▁diesen', '▁kompl', 'iz', 'ierten', '▁S', 'atz', '▁nicht', '.']


### Aufgabe 2 — Fertility messen

**Fertility** = durchschnittliche Anzahl Subword-Tokens pro (leerzeichen-getrenntem) Wort.
Sie misst, wie stark ein Vokabular eine Sprache „zerstückelt": 1.0 = jedes Wort bleibt ein
Token, höher = mehr Zerlegung.

**Deine Aufgabe (`# TODO`):** Implementiere `fertility(sentences)`:
Summe der Subword-Token über alle Sätze geteilt durch die Summe der Wörter
(`text.split()`). Berechne sie für Englisch und Deutsch getrennt und vergleiche.


In [4]:
# ---- AUFGABE 2: fertility ------------------------------------------------
def fertility(sentences):
    n_pieces = sum(len(sp.encode(s, out_type=str)) for s in sentences)
    n_words = sum(len(s.split()) for s in sentences)
    return n_pieces / n_words

# auf einer Auswertungs-Teilmenge (nicht die ersten -> etwas Varianz)
eval_en = [en for en, de in pairs[100000:105000]]
eval_de = [de for en, de in pairs[100000:105000]]
f_en, f_de = fertility(eval_en), fertility(eval_de)
print(f"Fertility EN (gemeinsames Vokabular): {f_en:.3f}")
print(f"Fertility DE (gemeinsames Vokabular): {f_de:.3f}")


Fertility EN (gemeinsames Vokabular): 1.500
Fertility DE (gemeinsames Vokabular): 1.433


### Aufgabe 3 — Vokabular-Teilung & Vokabular-Bias

**(a) Teilung:** In einem *gemeinsamen* Vokabular teilen sich beide Sprachen Tokens
(Ziffern, Interpunktion, gemeinsame Stämme/Kognaten). Miss den Anteil der Vokabular-Stücke,
die **sowohl** in englischen **als auch** in deutschen Sätzen tatsächlich vorkommen.

**(b) Bias-Experiment (`# TODO`):** Trainiere ein **englisch-dominiertes** Vokabular
(nur auf EN-Sätzen) und miss die deutsche fertility damit. Erwartung: Deutsch wird
**stärker zerstückelt** (höhere fertility) — genau der Nachteil, den ressourcenarme
Sprachen in englisch-zentrierten LLMs erfahren.


In [5]:
# ---- (a) Vokabular-Teilung (vorgegeben) ----------------------------------
def used_pieces(sentences):
    used = set()
    for s in sentences:
        used.update(sp.encode(s, out_type=str))
    return used

en_pieces = used_pieces([en for en, de in subset[:5000]])
de_pieces = used_pieces([de for en, de in subset[:5000]])
shared = en_pieces & de_pieces
print(f"Genutzte Stuecke EN: {len(en_pieces)},  DE: {len(de_pieces)}")
print(f"Geteilt (in beiden): {len(shared)}  "
      f"= {len(shared)/len(en_pieces | de_pieces):.1%} der genutzten Stuecke")
print("Beispiele geteilter Stuecke:", sorted(list(shared))[:20])


Genutzte Stuecke EN: 1583,  DE: 2532
Geteilt (in beiden): 301  = 7.9% der genutzten Stuecke
Beispiele geteilter Stuecke: ['!', '%', "'", ',', '.', '0', '1', '9', ':45', '?', 'N', 'a', 'ab', 'ack', 'ag', 'al', 'am', 'and', 'ant', 'ap']


In [6]:
# ---- (b) AUFGABE 3: Vokabular-Bias (EN-only-Vokabular) -------------------
EN_CORPUS = os.path.join(DATA_DIR, "corpus_en.txt")
with open(EN_CORPUS, "w", encoding="utf-8") as f:
    for en in en_sents:
        f.write(en + "\n")

EN_PREFIX = os.path.join(DATA_DIR, "en_only_bpe")
if not os.path.exists(EN_PREFIX + ".model"):
    spm.SentencePieceTrainer.train(
        input=EN_CORPUS, model_prefix=EN_PREFIX,
        vocab_size=8000, model_type="bpe",
        character_coverage=1.0, input_sentence_size=200000,
        shuffle_input_sentence=True)
sp_en = spm.SentencePieceProcessor(model_file=EN_PREFIX + ".model")

# TODO: fertility fuer EN und DE mit DIESEM (EN-only) Modell messen.
# Tipp: kleine Hilfsfunktion analog zu Aufgabe 2, aber mit sp_en statt sp.
def fertility_with(model, sentences):
    n_pieces = sum(len(model.encode(s, out_type=str)) for s in sentences)
    n_words = sum(len(s.split()) for s in sentences)
    return n_pieces / n_words

fe_en = fertility_with(sp_en, eval_en)
fe_de = fertility_with(sp_en, eval_de)
print(f"EN-only-Vokabular  -> Fertility EN: {fe_en:.3f},  DE: {fe_de:.3f}")
print(f"Gemeinsames Vokab. -> Fertility EN: {f_en:.3f},  DE: {f_de:.3f}")
print(f"\nDeutsch wird vom EN-only-Vokabular um Faktor {fe_de/f_de:.2f} staerker zerlegt.")


EN-only-Vokabular  -> Fertility EN: 1.430,  DE: 2.869
Gemeinsames Vokab. -> Fertility EN: 1.500,  DE: 1.433

Deutsch wird vom EN-only-Vokabular um Faktor 2.00 staerker zerlegt.


### Zum Vergleich — warum nicht Wörter? *(vorgegeben)*

Ein wortbasiertes Vokabular müsste jedes Wort speichern und scheitert an OOV. Die Zelle
zeigt die Vokabulargröße bei reiner Wort-Tokenisierung und die **OOV-Rate** auf ungesehenen
Sätzen — das Argument für Subwords aus Skript 2.1.


In [7]:
# ---- Wort-Tokenisierung: Vokabulargroesse & OOV (vorgegeben) --------------
from collections import Counter
train_words = Counter(w for en, de in subset for w in (en + " " + de).lower().split())
word_vocab = set(train_words)
print(f"Wort-Vokabular (DE+EN, {N:,} Paare): {len(word_vocab):,} Typen")

test_tokens = [w for en, de in pairs[200000:205000] for w in (en + " " + de).lower().split()]
oov = sum(1 for w in test_tokens if w not in word_vocab)
print(f"OOV-Rate auf ungesehenen Saetzen: {oov/len(test_tokens):.1%}")
print(f"Zum Vergleich: das BPE-Modell hat NIE OOV (Vokabular = {sp.get_piece_size()}, "
      f"faellt notfalls auf Zeichen/Bytes zurueck).")


Wort-Vokabular (DE+EN, 40,000 Paare): 20,591 Typen
OOV-Rate auf ungesehenen Saetzen: 11.3%
Zum Vergleich: das BPE-Modell hat NIE OOV (Vokabular = 8000, faellt notfalls auf Zeichen/Bytes zurueck).


## Reflexion (kurz, schriftlich)

1. **Subword vs. Wort:** Wie groß war das Wort-Vokabular und die OOV-Rate — und warum hat
   das BPE-Modell mit nur 8000 Stücken *keine* OOV-Wörter?

2. **Fertility:** Vergleiche EN und DE beim gemeinsamen Vokabular. Auf diesen kurzen
   Tatoeba-Sätzen liegen beide nah beieinander. Warum würde man bei *längeren*,
   kompositareichen deutschen Texten (z. B. „Donaudampfschifffahrtsgesellschaft") eine
   deutlich höhere deutsche fertility erwarten?

3. **Vokabular-Bias:** Um welchen Faktor stieg die deutsche fertility mit dem
   EN-only-Vokabular? Erkläre, warum das für ressourcenarme Sprachen ein **Kosten**- und
   **Fairness**-Problem ist (mehr Tokens ⇒ längere Sequenzen ⇒ mehr Rechenzeit/Geld, und
   das Kontextfenster reicht für weniger Text).

4. **Teilung:** Welche Art von Stücken teilen sich Deutsch und Englisch typischerweise
   (schau in die Beispiel-Liste)? Warum ist so eine Teilung die Grundlage für
   cross-lingualen Transfer (Ausblick auf Projekt 02 & 03)?

> **Referenz** (gemeinsames Vokab, kurze Tatoeba-Sätze): fertility EN ≈ 1.5, DE ≈ 1.4;
> Wort-Vokabular ~20k Typen mit ~11 % OOV; das EN-only-Vokabular **verdoppelt** die
> deutsche fertility (≈ 2.9, Faktor ~2.0). Genaue Zahlen variieren leicht mit der Teilmenge.
